# 1-2. 토큰화 심화

## 학습 목표
- BPE / WordPiece / SentencePiece 알고리즘의 차이를 설명할 수 있다.
- `tiktoken`으로 동일 의미의 영·한 문장의 토큰 수 차이를 측정한다.
- Llama3 / Qwen2.5 / bert-multilingual 토크나이저의 한국어 효율을 벤치마크한다.
- 토큰 효율 차이가 실제 API 비용에 주는 영향을 계산한다.

## 핵심 키워드
`BPE`, `WordPiece`, `SentencePiece`, `어휘 크기`, `OOV`, `subword`, `토큰 연비(fertility)`

## 먼저 한 장으로 감 잡기

- **토큰이란?** — LLM이 다루는 최소 단위. 단어보다 작은 "조각(subword)"인 경우가 많다.
  - 예: `tokenization` → `token` + `ization` (2조각)
  - 예: `"금융"` → `cl100k`에서는 바이트 4개로 쪼개지고, `o200k`에서는 1조각으로 유지.
- **왜 subword인가?** — 단어 단위로 하면 어휘가 폭발하고(OOV↑), 문자 단위로 하면 시퀀스가 너무 길어진다. 중간 타협.
- **"토큰 연비(fertility)"** — 한 단어 또는 한 글자를 몇 개의 토큰으로 쪼개는가? **낮을수록 효율적**.

### 한국어 토큰화 맛보기 🍱
```
영어 "banking"                   → ['bank', 'ing']                       2 tokens
한국어 "전자금융거래법"           → ['전자', '금', '융', '거', '래', '법']   6 tokens  (o200k)
                                → ['전', '자', '금', '융', '거', '래', '법']  더 쪼개짐 (cl100k)
```

같은 의미라도 한국어 쪽이 토큰을 **1.3~1.8배** 더 쓴다 → 그대로 비용·속도·컨텍스트 낭비로 이어진다. 이 노트북의 핵심 메시지.

## 0. 왜 토큰화가 중요한가?

LLM은 사람이 쓰는 문자 그대로가 아니라 **토큰 ID의 정수 시퀀스**로 입출력을 처리한다. 그래서 같은 문장이라도 어떤 토크나이저를 쓰느냐에 따라 실제로 다뤄지는 길이가 달라진다. 이 길이가 **곧 비용/속도/컨텍스트**를 결정한다.

| 영향 | 이유 |
|---|---|
| **💰 비용** | OpenAI / Anthropic API는 **토큰당** 과금. 같은 메시지라도 토큰이 많은 모델이 비쌈 |
| **⏱ 속도** | 생성은 토큰 단위 autoregressive → 토큰이 많을수록 지연(latency)이 늘어남 |
| **📏 컨텍스트 윈도우** | 4K · 128K 같은 상한이 **토큰 기준** → 토큰 효율 나쁜 모델은 실데이터를 덜 담음 |
| **🇰🇷 한국어 패널티** | 영어 중심 초기 토크나이저는 한글을 **UTF-8 바이트 2~3개 단위**로 잘게 쪼갬 |

이번 노트북에서는 이 차이를 실제 숫자로 눈으로 확인한다. 뒤에서 나오는 `�` 표시는 **버그가 아니라** 이 "잘게 쪼갬"의 흔적이므로 미리 알아두자 (3.1절에서 자세히).

## 1. 토큰화 알고리즘 배경

BPE · WordPiece · SentencePiece는 모두 **통계적 subword 알고리즘** 계열. 공통 아이디어: "자주 함께 등장하는 문자 쌍을 **하나의 토큰으로 병합**하자."

| 알고리즘 | 핵심 아이디어 | 대표 모델 |
|---|---|---|
| **BPE** (Byte-Pair Encoding) | 가장 자주 등장하는 문자 쌍을 병합 | GPT-2, GPT-3, GPT-4, Llama |
| **WordPiece** | likelihood 가 증가하는 쌍을 병합, 접두어·추가 subword는 `##` 접두사 | BERT, DistilBERT |
| **SentencePiece** (BPE/Unigram) | 공백을 특수 문자 `▁`로 대체해 언어 독립적, 디토크나이제이션 되돌림 | T5, ALBERT, XLM-R, Llama2, Gemma |
| **Tiktoken BPE (cl100k/o200k)** | BPE를 byte 레벨에 적용 (UTF-8 raw byte) → OOV 없음 | GPT-3.5/4/4o |

### 🧑‍🏫 BPE 한 번에 이해하기 (손으로 따라가는 예)

학습 코퍼스에 단어 4개가 있다고 하자. `</w>`는 단어 끝을 뜻한다.

```
freq: {"low</w>": 5, "lower</w>": 2, "newest</w>": 6, "widest</w>": 3}
```

초기 vocab은 모든 **문자**. 이제 가장 자주 나온 **인접 쌍**을 찾아 병합한다.

| 반복 | 가장 흔한 쌍 | 등장 횟수 | 병합 후 신규 토큰 |
|---|---|---|---|
| 1 | `(e, s)` | 6+3 = 9 | **es** → "new**es**t", "wid**es**t" |
| 2 | `(es, t)` | 9 | **est** → "new**est**", "wid**est**" |
| 3 | `(est, </w>)` | 9 | **est</w>** |
| 4 | `(l, o)` | 7 | **lo** |
| 5 | `(lo, w)` | 7 | **low** |
| … | | | |

이렇게 자주 나오는 조각이 점점 한 토큰으로 묶여 **어휘 크기 V** 에 도달하면 멈춘다. 결과로 `low`, `lower`, `newest`, `widest` 같은 단어가 `low` + `</w>` / `low` + `er</w>` / `new` + `est</w>` / `wid` + `est</w>` 로 **공유된 조각**으로 표현된다.

### 알고리즘 의사코드
```text
BPE:
  1. 모든 단어를 문자 단위로 분해.                     ("low </w>") → l o w </w>
  2. 인접 쌍을 세고, 빈도가 가장 높은 쌍을 병합.         (l, o) → "lo"
  3. 어휘 크기(V)에 도달할 때까지 반복.

WordPiece:
  1. BPE처럼 문자 단위로 시작.
  2. 모든 후보 쌍 (a, b)에 대해 score(ab) = count(ab) / (count(a)*count(b))
  3. score 가 가장 큰 쌍을 병합.                       ← "함께가 자주, 따로가 드문" 쌍 우선
```

### 한국어에서 왜 자소 단위로 쪼개지는가?
- 영어 중심 코퍼스로 BPE를 학습하면, 한국어 음절(`금`, `융` 등)은 **함께 등장 빈도가 낮아** 거의 병합되지 않는다.
- 그 결과 한국어가 나올 때는 **UTF-8 byte 단위**까지 내려가 쪼개진다 (한 글자가 3 byte → 최대 3 토큰으로 분해).
- 이게 3.1절의 `�` 표시로 나타날 현상이다.

In [1]:
import sys
sys.path.insert(0, '../..')

# 준비
import tiktoken
from transformers import AutoTokenizer

print('tiktoken OK')
print('transformers OK')

tiktoken OK
transformers OK


## 2. tiktoken으로 영/한 토큰 수 비교

OpenAI의 **tiktoken**은 GPT 계열 모델의 실제 토크나이저를 그대로 쓸 수 있는 파이썬 라이브러리다. 모델별 기본 인코딩:

| 인코딩 | 대응 모델 | 비고 |
|---|---|---|
| `o200k_base` | GPT-4o, GPT-4o-mini | 한국어 merges 대폭 추가 |
| `cl100k_base` | GPT-4, GPT-3.5-turbo | 영어 중심 초기 어휘 |
| `p50k_base` | GPT-3 (davinci 계열) | 옛 버전 |

아래에서 **의미가 똑같은** 영-한 문장 쌍을 인코딩하고 토큰 수 비율을 측정한다. 차이가 바로 돈이다.

In [2]:
enc_o200k = tiktoken.get_encoding('o200k_base')  # gpt-4o
enc_cl100k = tiktoken.get_encoding('cl100k_base')  # gpt-4 / 3.5-turbo

# 동일한 의미의 영·한 문장 쌍 (금융·행정 주제)
pairs = [
    ('The customer submitted an application for an electronic banking service.',
     '고객이 전자금융 서비스 신청서를 제출했습니다.'),
    ('Banks must protect personal information under the Personal Information Protection Act.',
     '은행은 개인정보보호법에 따라 개인정보를 보호해야 합니다.'),
    ('Network separation is a core principle of financial security.',
     '망분리는 금융 보안의 핵심 원칙입니다.'),
    ('Large language models are based on the Transformer architecture.',
     '대규모 언어모델은 트랜스포머 구조를 기반으로 합니다.'),
    ('Retrieval-Augmented Generation reduces hallucination by grounding answers in documents.',
     '검색 증강 생성은 답변을 문서에 근거해 환각을 줄입니다.'),
    ('The airgapped environment physically isolates internal networks from the Internet.',
     '폐쇄망 환경은 내부망을 인터넷과 물리적으로 분리합니다.'),
    ('Knowledge graphs represent entities and relations as nodes and edges.',
     '지식 그래프는 개체와 관계를 노드와 엣지로 표현합니다.'),
]

print(f'{"EN tok":>8} {"KO tok":>8} {"ratio":>8}  text')
print('-' * 70)
total_en, total_ko = 0, 0
for en, ko in pairs:
    en_tok = len(enc_o200k.encode(en))
    ko_tok = len(enc_o200k.encode(ko))
    total_en += en_tok
    total_ko += ko_tok
    print(f'{en_tok:>8} {ko_tok:>8} {ko_tok / en_tok:>8.2f}x  {ko[:30]}...')

print('-' * 70)
print(f'total   EN={total_en}  KO={total_ko}  ratio={total_ko / total_en:.2f}x')

  EN tok   KO tok    ratio  text
----------------------------------------------------------------------
      11       13     1.18x  고객이 전자금융 서비스 신청서를 제출했습니다....
      12       14     1.17x  은행은 개인정보보호법에 따라 개인정보를 보호해야 합니다...
      10       13     1.30x  망분리는 금융 보안의 핵심 원칙입니다....
      10       19     1.90x  대규모 언어모델은 트랜스포머 구조를 기반으로 합니다....
      14       20     1.43x  검색 증강 생성은 답변을 문서에 근거해 환각을 줄입니다...
      13       17     1.31x  폐쇄망 환경은 내부망을 인터넷과 물리적으로 분리합니다....
      11       20     1.82x  지식 그래프는 개체와 관계를 노드와 엣지로 표현합니다....
----------------------------------------------------------------------
total   EN=81  KO=116  ratio=1.43x


해석
- 동일 의미라도 **GPT-4o 기준** 한국어가 영어 대비 1.3~1.8배 토큰을 쓴다 (o200k).
- 초기 `cl100k`(GPT-4/3.5)에서는 **2.5배 이상**이었음 → o200k가 상당히 개선한 셈.
- 즉, 한국어 프롬프트를 GPT-3.5에서 GPT-4o-mini로 올리는 것만으로도 **1토큰당 비용은 같아도 총 토큰 수가 줄어 실제 요금이 ~40% 감소** 하는 효과가 난다.

In [3]:
# cl100k 와 o200k 직접 비교 (동일 한국어 문장)
ko_texts = [p[1] for p in pairs]
cl100k_total = sum(len(enc_cl100k.encode(t)) for t in ko_texts)
o200k_total = sum(len(enc_o200k.encode(t)) for t in ko_texts)
print(f'cl100k (GPT-4 / 3.5-turbo) : {cl100k_total} tokens')
print(f'o200k  (GPT-4o)           : {o200k_total} tokens')
print(f'개선율: {(1 - o200k_total / cl100k_total) * 100:.1f}% 줄어듦')

cl100k (GPT-4 / 3.5-turbo) : 205 tokens
o200k  (GPT-4o)           : 116 tokens
개선율: 43.4% 줄어듦


## 3. 토큰화 시각적 확인

한국어가 실제로 어떻게 쪼개지는지 본다.

In [ ]:
def show_tokens(encoder, text, label):
    """문자열 → 토큰 ID → 각 ID를 decode 해 보기 쉽게 출력."""
    ids = encoder.encode(text)
    pieces = [encoder.decode([i]) for i in ids]
    print(f'[{label}] {len(ids)} tokens')
    print('  →', ' | '.join(pieces))


sentence_ko = '전자금융거래법은 금융소비자를 보호한다.'
show_tokens(enc_o200k, sentence_ko, 'o200k (gpt-4o)')
show_tokens(enc_cl100k, sentence_ko, 'cl100k (gpt-4)')

sentence_en = 'The Electronic Financial Transactions Act protects financial consumers.'
show_tokens(enc_o200k, sentence_en, 'o200k (gpt-4o) EN')

### 3.1 왜 `�` (네모) 가 뜨는가? — 버그가 아닙니다

위 `cl100k` 출력에서 `|  � | � | � | � | 소 | 비 | ...` 처럼 **유니코드 복제문자(U+FFFD)** 가 보인다. 터미널이나 폰트 문제가 아니라 **토크나이저가 한 글자를 바이트 중간까지 쪼갰다**는 증거다.

한글 한 글자는 UTF-8에서 **3 byte** 로 저장된다. `cl100k`의 BPE 어휘는 영어 중심이라 한글 음절을 하나의 토큰으로 가지고 있지 않고 대신 **자주 등장하는 2-byte 패턴**을 토큰으로 가진다. 그래서 "금" (`\xea\xb8\x88`) 을 토큰화하면:

```
금 = 3 bytes: \xea \xb8 \x88
   → cl100k: [\xea\xb8] + [\x88]  ← 두 개의 토큰(불완전 UTF-8)
```

각 토큰을 개별로 `decode` 하면 유효한 UTF-8이 아니므로 파이썬이 `�` 로 대체 표시한다. 하지만 **토큰 리스트 전체를 한 번에 decode 하면** 정상 한글로 복원된다. 아래 셀이 이 상황을 직접 보여준다.

In [ ]:
def show_tokens_with_bytes(encoder, text, label):
    """각 토큰의 원본 bytes와 decode 결과를 동시에 보여주는 확장 버전."""
    ids = encoder.encode(text)
    raw_bytes = encoder.decode_tokens_bytes(ids)  # 각 ID의 원본 bytes
    full = encoder.decode(ids)                    # 전체를 한 번에 decode
    print(f'[{label}] {len(ids)} tokens')
    print(f'  전체 decode (정상):   {full!r}')
    print(f'  토큰별 원본 bytes와 단일 decode 결과:')
    for i, (tid, b) in enumerate(zip(ids, raw_bytes)):
        # 개별 decode — 불완전 UTF-8은 replacement '�' 로 보임
        try:
            single = b.decode('utf-8')
        except UnicodeDecodeError:
            single = b.decode('utf-8', errors='replace') + '  ← 불완전 바이트'
        print(f'    [{i:2d}] id={tid:>7}  bytes={b!r:<22} decode={single!r}')


# 바로 위에서 봤던 "금"의 수수께끼를 해부
show_tokens_with_bytes(enc_cl100k, '금', 'cl100k · 글자 "금"')
print()
show_tokens_with_bytes(enc_o200k, '금', 'o200k · 글자 "금"')
print()
# 전체 문장에서도 확인 — 전체 decode는 깨지지 않음을 보자
print('─' * 60)
print('전체 decode는 언제나 정상:')
ids = enc_cl100k.encode('전자금융거래법')
print(f'  cl100k 토큰 수: {len(ids)} ids, decode: {enc_cl100k.decode(ids)!r}')
ids = enc_o200k.encode('전자금융거래법')
print(f'  o200k  토큰 수: {len(ids)} ids, decode: {enc_o200k.decode(ids)!r}')

## 4. 오픈소스 토크나이저 한국어 효율 벤치마크

폐쇄망에 배치하려는 모델 후보들에 대해 한국어 토큰 효율을 측정한다. 토크나이저는 모델과 한 쌍이므로, 모델 선정 전 **토큰 효율을 먼저 측정하면** 컨텍스트 사용량과 추론 속도를 예측할 수 있다.

### 지표
- **토큰/문자 비율** (**낮을수록 좋음**): 1글자를 몇 토큰에 담는가
- **문자/토큰 비율** (= fertility 역수, **높을수록 좋음**): 1토큰이 몇 글자를 담는가

> 직관: 한국어 `chars/tok ≈ 1.0` 이면 "한 글자당 한 토큰" — 비효율. `≈ 2.5` 이면 짧은 단어가 한 토큰에 들어감 — 효율적.

In [ ]:
# 샘플 코퍼스 로드 — 전자금융거래·개인정보보호법 관련 문장들 (UTF-8)
from pathlib import Path
corpus = Path('../../data/corpus_ko.txt').read_text(encoding='utf-8')
print('전체 문자 수:', len(corpus))
print('아래는 첫 200자:\n', corpus[:200], '...')

In [ ]:
# 모델별 토크나이저 로드 (실행 시 최초에 다운로드 발생)
# 폐쇄망 에서는 사전 다운로드된 모델을 사용.
TOKENIZERS_TO_COMPARE = {
    'bert-base-multilingual-cased': 'bert-base-multilingual-cased',
    'Llama-3.1-8B (proxy)': 'NousResearch/Meta-Llama-3.1-8B',  # gated이라 대체 레포
    'Qwen2.5-7B': 'Qwen/Qwen2.5-7B',
    'xlm-roberta-base': 'xlm-roberta-base',
}

loaded = {}
for label, model_id in TOKENIZERS_TO_COMPARE.items():
    try:
        loaded[label] = AutoTokenizer.from_pretrained(model_id)
        print(f'OK  {label:<35}  vocab={loaded[label].vocab_size}')
    except Exception as e:
        print(f'SKIP {label:<35}  ({type(e).__name__})')

In [ ]:
# 벤치마크
results = []
n_chars = len(corpus)

# tiktoken 도 비교에 포함
for name, enc in [('tiktoken-o200k (gpt-4o)', enc_o200k), ('tiktoken-cl100k (gpt-4)', enc_cl100k)]:
    n_tokens = len(enc.encode(corpus))
    results.append((name, n_tokens, n_chars / n_tokens))

for label, tok in loaded.items():
    n_tokens = len(tok.encode(corpus, add_special_tokens=False))
    results.append((label, n_tokens, n_chars / n_tokens))

print(f'{"tokenizer":<40} {"tokens":>8} {"chars/tok":>12}')
print('-' * 64)
for name, n_tok, ratio in sorted(results, key=lambda r: -r[2]):
    print(f'{name:<40} {n_tok:>8} {ratio:>12.3f}')

print()
print('* chars/tok 가 클수록 한국어 효율이 높음')

해석 가이드

- BERT multilingual: 어휘가 11만이라 한국어 자소가 많아 효율이 낮다.
- Llama3 / Qwen2.5: 어휘 12만~15만 수준 + 다국어 데이터 학습 → cl100k 대비 한국어가 효율적.
- tiktoken o200k: GPT-4o 용. 한국어 merges를 대폭 추가해 cl100k 대비 30~40% 개선.
- xlm-roberta: 원래 다국어 용으로 학습되어 한국어 어휘가 많은 편.

## 5. 실제 비용 계산

gpt-4o-mini 요금 (2024-11 기준):
- Input: $0.150 / 1M tokens
- Output: $0.600 / 1M tokens

전자금융감독규정 전체 본문(약 15만자)을 context에 넣는 시나리오.

In [ ]:
CORPUS_CHARS = 150_000  # 규정 전체 문자 수 (가정)
OUTPUT_TOKENS = 500     # 요약 답변

# 위에서 계산된 chars/tok 비율을 사용
tok_ratios = {name: ratio for name, _, ratio in results}

PRICING_PER_M = {'gpt-4o-mini': (0.150, 0.600)}  # (input, output)

input_per_m, output_per_m = PRICING_PER_M['gpt-4o-mini']
print(f'{"tokenizer":<40} {"input tok":>10} {"$/req":>10}')
print('-' * 64)
for name in ['tiktoken-o200k (gpt-4o)', 'tiktoken-cl100k (gpt-4)']:
    if name not in tok_ratios:
        continue
    chars_per_tok = tok_ratios[name]
    input_tokens = int(CORPUS_CHARS / chars_per_tok)
    cost = (input_tokens * input_per_m + OUTPUT_TOKENS * output_per_m) / 1_000_000
    print(f'{name:<40} {input_tokens:>10} ${cost:>9.4f}')

print()
print('시나리오: 1만 번 시행하면 o200k 기준 전체 비용?')
chars_per_tok = tok_ratios['tiktoken-o200k (gpt-4o)']
input_tokens = int(CORPUS_CHARS / chars_per_tok)
batch_cost = 10_000 * (input_tokens * input_per_m + OUTPUT_TOKENS * output_per_m) / 1_000_000
print(f'=> ${batch_cost:,.2f}')

<!-- optional -->

## 6. 실전 팁: 한국어 토큰 절약 기술

1. **불필요한 공백·줄바꿈 제거** — PDF에서 추출한 텍스트는 공백이 매우 많을 수 있다.
2. **긴 숫자 · 테이블 → markdown 변환** — BPE는 숫자를 자릿수 단위로 쪼개는 경향이 있다.
3. **프롬프트 영어 컨버전** — system prompt를 영어로 쓰면 지시부 토큰 40~60% 절감 가능. 단, 한국어 답변 요구는 명시.
4. **cached input 활용** — OpenAI 기준 cached prompt는 입력 비용 50% 할인.

In [ ]:
# <!-- optional -->
# 영어 system prompt 대 한국어 system prompt 토큰 수 비교
sys_ko = ('당신은 한국어 금융 규제 전문가입니다. 정확하고 간결하게 답변하고, '
          '법령 조문 번호를 반드시 인용하며, 불확실한 정보는 추측하지 않습니다.')
sys_en = ('You are a Korean financial regulation expert. Answer accurately and concisely, '
          'cite statute numbers, and do not speculate on uncertain facts.')
print(f'KO system prompt: {len(enc_o200k.encode(sys_ko))} tokens')
print(f'EN system prompt: {len(enc_o200k.encode(sys_en))} tokens')


## 더 읽어보기
- Sennrich et al., [Neural Machine Translation of Rare Words with Subword Units (BPE, 2016)](https://arxiv.org/abs/1508.07909)
- Kudo & Richardson, [SentencePiece: A simple and language independent subword tokenizer (2018)](https://arxiv.org/abs/1808.06226)
- [OpenAI Tokenizer Playground](https://platform.openai.com/tokenizer)